# Actualizar un usuario

Cambia el nombre, el email o ambos de un usuario que ya existe. Lo que no envíes se queda como estaba.

**Qué pasa en la base:** ⚠️ **usa la base de datos real** (`src/db/users.sqlite`) y **modifica un usuario** y su fecha `updatedAt`. No hay forma de deshacerlo.

**Acción que usa:** `PATCH http://127.0.0.1:3004/users/:id` con el cuerpo `{"name": ..., "email": ...}`. Responde `200` con el usuario ya cambiado.

| Dato | Significado | ¿Obligatorio? | Ejemplo válido |
|---|---|---|---|
| `user_id` | Número del usuario (míralo en *Listar usuarios*) | Sí | `3` |
| `name` | Nombre nuevo. `None` para no cambiarlo | No, pero hace falta `name` o `email` | `"Lin Chen"` |
| `email` | Email nuevo, que no use otro usuario. `None` para no cambiarlo | No, pero hace falta `name` o `email` | `"lin.chen@example.com"` |

**Anotaciones:** necesitas `pnpm start` y `pnpm node-http` en marcha. El `user_id` empieza en `0`, que no existe: si pulsas **▶ Run** sin cambiar nada saldrá `No user was found.` y no se modifica a nadie. Cambia el `user_id` por uno real y el `name` o el `email` marcados con ✏️. **Cómo reconocer que ha funcionado:** sale una tabla con el usuario y sus datos nuevos. Si falla, sale solo el mensaje y ninguna tabla.

- `No user was found.` (404): ningún usuario tiene ese id. Usa uno de *Listar usuarios*.
- `There is nothing to update.` (400): `name` y `email` están en `None`. Indica al menos uno.
- `The name is missing.` (400): el nombre nuevo está vacío. Escribe un nombre, o déjalo en `None`.
- `The email is not valid.` (400): el email nuevo no tiene algo antes y después de la `@`. Escríbelo completo.
- `This email is already used.` (409): otro usuario ya tiene ese email. Elige uno distinto.
- **Con la base parada:** para el programa de la base de datos (`pnpm start`) y vuelve a pulsar **▶ Run**: sale `The database failed.` y ninguna tabla.
- Si sale `The HTTP client is not running. Start it with pnpm node-http.`, arranca el cliente en otra terminal (con `pnpm start` ya en marcha) y repite.

[← Anterior: Buscar por id](find-user-by-id.ipynb) · [Siguiente: Eliminar un usuario →](delete-user.ipynb)

In [ ]:
import json
import os
import subprocess
import urllib.error
import urllib.request

# ✏️ CAMBIA AQUÍ (user_id 0 no existe: no cambia a nadie)
user_id = 0
name = "Lin Chen"  # o None para no cambiarlo
email = None  # o un email nuevo; None para no cambiarlo

# Solo se envían los campos que se quieren cambiar
changes = {"name": name, "email": email}
changes = {key: value for key, value in changes.items() if value is not None}

URL = "http://127.0.0.1:3004/users/" + str(user_id)
request = urllib.request.Request(
    URL,
    data=json.dumps(changes).encode(),
    headers={"Content-Type": "application/json"},
    method="PATCH",
)

# Envía la petición al cliente. Una respuesta de error (400, 404, 409, 500) también trae un texto
status = None
text = ""
try:
    with urllib.request.urlopen(request) as answer:
        status = answer.status
        text = answer.read().decode()
except urllib.error.HTTPError as error:
    status = error.code
    text = error.read().decode()
except urllib.error.URLError:
    pass

# Código de Node que dibuja las filas en una tabla con cli-table3
js = """
import Table from "cli-table3";

const rows = JSON.parse(process.argv[1]);

const columns = ["id", "name", "email", "createdAt", "updatedAt"];
const table = new Table({ head: columns, style: { head: ["cyan", "bold"] } });

for (const row of rows) {
  table.push(columns.map((column) => row[column]));
}
console.log(table.toString());
"""

if status is None:
    print("The HTTP client is not running. Start it with pnpm node-http.")
elif status >= 400:
    # Un fallo: solo el mensaje, sin tabla
    print(json.loads(text)["error"])
else:
    data = json.loads(text)
    rows = [data]
    # Lanza Node desde la carpeta first-app (tres carpetas más arriba) y le pasa las filas como JSON
    result = subprocess.run(
        ["node", "--disable-warning=ExperimentalWarning", "--input-type=module", "-e", js, json.dumps(rows)],
        cwd="../../..",
        env={**os.environ, "FORCE_COLOR": "1"},  # conserva el color de los encabezados
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr)